# Experimental Run — GEO Audit

Pipeline experimental que mide la visibilidad GEO de programamos.es.

**Prerequisito**: Haber ejecutado `00_discover_competitors.ipynb` para tener
el vectorstore congelado en `data/frozen_vectorstore/`.

**Que hace cada run**:
1. Carga el vectorstore congelado de competidores
2. Scrapea el contenido ACTUAL de programamos.es
3. Genera embeddings solo del target y los mezcla con los congelados
4. Para cada query: retrieve → RAG Judge (JSON) → Citation Extractor
5. Guarda scorecard JSON en `data/geo/experimental/`

Ver ADR-006 en `docs/DECISIONS.md`

In [ ]:
# === Bootstrap Kaggle ===
import os, sys

if os.path.exists('/kaggle'):
    # 1. Clonar / actualizar repo
    REPO_DIR = "/kaggle/working/TFG"

    if os.path.exists(REPO_DIR):
        print("Repo ya existe, haciendo pull...")
        !cd {REPO_DIR} && git pull
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("github_token")
        REPO_URL = f"https://{token}@github.com/angelmanuelferrer/TFG.git"
        print("Clonando repo...")
        !git clone {REPO_URL} {REPO_DIR}

    # 2. Instalar dependencias
    %pip install -q \
        "langchain>=0.3,<0.4" \
        "langchain-community>=0.3,<0.4" \
        "langchain-huggingface>=0.1,<1" \
        "langchain-openai>=0.3,<0.4" \
        "langchain-google-genai>=4,<5" \
        "langgraph>=0.3,<0.4" \
        "openai>=1,<2" \
        "google-genai>=1,<2" \
        "faiss-cpu>=1.9,<2" \
        "tiktoken>=0.8,<1" \
        "sentence-transformers>=3,<4" \
        "beautifulsoup4>=4,<5" \
        "requests>=2,<3" \
        "python-dotenv>=1,<2"

    # 3. Configurar entorno
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
    os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'

    print(f"Entorno Kaggle listo — {REPO_DIR}")
else:
    from dotenv import load_dotenv
    load_dotenv()
    sys.path.insert(0, os.path.abspath('..'))

from src.config import PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Cargar configuracion ===
import json
from datetime import datetime
from src.config import load_experiment_config, get_queries_for_run

config = load_experiment_config()
target_url = config['target_url']
target_brand = config['target_brand']

# === Seleccion de bloque rotativo ===
# Core 20 queries se ejecutan siempre.
# Opcionalmente anadir un bloque rotativo: "R1", "R2", "R3", "R4", o None
ROTATION_BLOCK = "R1"  # Cambiar segun el run

queries = get_queries_for_run(block=ROTATION_BLOCK)

RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
print(f"Run: {RUN_ID}")
print(f"Target: {target_url}")
print(f"Bloque rotativo: {ROTATION_BLOCK}")
print(f"Queries: {len(queries)} (20 core + {len(queries)-20} rotativas)")

In [ ]:
# === Cargar vectorstore congelado de competidores ===
from src.processing.embeddings import create_embeddings
from langchain_community.vectorstores import FAISS

embeddings = create_embeddings(config)
vs_path = PROJECT_ROOT / 'data' / 'frozen_vectorstore'

if not vs_path.exists():
    raise FileNotFoundError(
        f"Vectorstore congelado no encontrado en {vs_path}/\n"
        "Ejecuta 00_discover_competitors.ipynb primero."
    )

competitor_vs = FAISS.load_local(
    str(vs_path), embeddings, allow_dangerous_deserialization=True
)
print(f"Vectorstore congelado cargado: {competitor_vs.index.ntotal} vectores")

In [ ]:
# === Crawl contenido actual de programamos.es (sitio completo) ===
from src.processing.site_crawler import SiteCrawler
from src.processing.chunker import TokenAwareChunker

crawler = SiteCrawler(config)
target_docs = crawler.crawl_site(target_url, is_target=True)

if not target_docs:
    raise RuntimeError(f"No se pudo crawlear {target_url}")

chunker = TokenAwareChunker(config)
target_chunks = chunker.chunk_documents(target_docs)
print(f"Target: {len(target_docs)} pages -> {len(target_chunks)} chunks de programamos.es")

In [ ]:
# === Merge vectores: target actual + competidores congelados ===
target_vs = FAISS.from_documents(target_chunks, embeddings)
target_vs.merge_from(competitor_vs)

retriever = target_vs.as_retriever(
    search_kwargs={'k': config['retrieval']['top_k']}
)
print(f"Vectorstore combinado: {target_vs.index.ntotal} vectores total")

In [ ]:
# === Ejecutar RAG Judge (agent mode) + Citation Extractor para cada query ===
import time
from src.rag.judge import RAGJudge
from src.rag.citation_extractor import CitationExtractor

judge = RAGJudge(config)
extractor = CitationExtractor(target_url, target_brand)

# En modo agente, el judge usa el vectorstore directamente (decide que buscar)
use_agent = config.get("rag_simulator", {}).get("mode") == "agent"

# Delay entre queries para respetar rate limit (Gemini free tier = 15 RPM).
# El agente hace ~2-3 llamadas LLM por query, asi que 5s entre queries
# da ~12 RPM efectivo con margen de seguridad.
QUERY_DELAY = 5.0

results = []
for i, query in enumerate(queries, 1):
    print(f"\n[{i}/{len(queries)}] {query}")
    
    try:
        if use_agent:
            judge_output = judge.generate_answer_with_agent(query, target_vs)
        else:
            retrieved = retriever.invoke(query)
            judge_output = judge.generate_answer(query, retrieved)
        
        metrics = extractor.extract_metrics(judge_output)
        
        result = {
            'query': query,
            'judge_output': judge_output,
            'metrics': metrics,
            'mode': 'agent' if use_agent else 'classic',
        }
        results.append(result)
        
        visible = '✓' if metrics['is_visible'] else '✗'
        print(f"  {visible} Visible={metrics['is_visible']}, SoM={metrics['som']}%, Rank={metrics['first_citation_rank']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({'query': query, 'error': str(e)})
    
    # Esperar entre queries para no saturar el rate limit
    if i < len(queries):
        time.sleep(QUERY_DELAY)

In [ ]:
# === Guardar scorecard ===
run_dir = PROJECT_ROOT / 'data' / 'geo' / 'experimental' / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)

# Scorecard resumen
valid_results = [r for r in results if 'metrics' in r]
visible_count = sum(1 for r in valid_results if r['metrics']['is_visible'])
ranked_results = [r for r in valid_results if r['metrics']['first_citation_rank'] is not None]

# PAWC: media sobre queries con citas del target (pawc > 0)
pawc_results = [r for r in valid_results if r['metrics']['pawc'] > 0]
avg_pawc = (
    sum(r['metrics']['pawc'] for r in pawc_results) / len(pawc_results)
    if pawc_results else 0.0
)

# Citation Rate: media sobre queries donde el target fue recuperado (cr != None)
cr_results = [r for r in valid_results if r['metrics']['citation_rate'] is not None]
avg_citation_rate = (
    sum(r['metrics']['citation_rate'] for r in cr_results) / len(cr_results)
    if cr_results else None
)

# Coverage: visibilidad desglosada por categoria de query
from src.config import load_queries
queries_data = load_queries()
queries_db = queries_data.get("queries", {})

# Mapeo texto → categoria
text_to_category = {q["text"]: q["category"] for q in queries_db.values()}

coverage = {}
for category in ("informacional", "comparativa", "navegacional"):
    cat_results = [
        r for r in valid_results
        if text_to_category.get(r['query']) == category
    ]
    if cat_results:
        cat_visible = sum(1 for r in cat_results if r['metrics']['is_visible'])
        coverage[category] = round((cat_visible / len(cat_results)) * 100, 2)
    else:
        coverage[category] = None

scorecard = {
    'run_id': RUN_ID,
    'timestamp': datetime.now().isoformat(),
    'target_url': target_url,
    'target_brand': target_brand,
    'n_queries': len(queries),
    'n_successful': len(valid_results),
    'aggregate_metrics': {
        'visibility_rate': (visible_count / len(valid_results) * 100) if valid_results else 0,
        'avg_som': sum(r['metrics']['som'] for r in valid_results) / len(valid_results) if valid_results else 0,
        'avg_rank': sum(r['metrics']['first_citation_rank'] for r in ranked_results) / len(ranked_results) if ranked_results else None,
        'avg_pawc': round(avg_pawc, 2),
        'avg_citation_rate': round(avg_citation_rate, 2) if avg_citation_rate is not None else None,
        'coverage': coverage,
    },
    'token_usage': judge.get_token_usage_summary(),
    'config_snapshot': config,
}

# Guardar archivos
with open(run_dir / 'scorecard.json', 'w', encoding='utf-8') as f:
    json.dump(scorecard, f, ensure_ascii=False, indent=2)

with open(run_dir / 'raw_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Resultados guardados en {run_dir}/")
print(f"\n=== RESUMEN ===")
print(f"Visibilidad: {scorecard['aggregate_metrics']['visibility_rate']:.1f}%")
print(f"SoM promedio: {scorecard['aggregate_metrics']['avg_som']:.1f}%")
avg_rank = scorecard['aggregate_metrics']['avg_rank']
print(f"Rank promedio: {f'{avg_rank:.1f}' if avg_rank else 'N/A (no citado)'}")
print(f"PAWC promedio: {scorecard['aggregate_metrics']['avg_pawc']:.1f}")
cr_display = f"{scorecard['aggregate_metrics']['avg_citation_rate']:.1f}%" if scorecard['aggregate_metrics']['avg_citation_rate'] is not None else 'N/A'
print(f"Citation Rate promedio: {cr_display}")
print(f"Coverage: {coverage}")
print(f"Tokens usados: {scorecard['token_usage']}")

In [ ]:
# === Detalle por query ===
print(f"{'Query':<60} {'Vis':>4} {'SoM':>6} {'Rank':>5} {'PAWC':>7} {'CR':>6}")
print('-' * 94)
for r in results:
    if 'metrics' in r:
        m = r['metrics']
        vis = 'SI' if m['is_visible'] else 'NO'
        rank = str(m['first_citation_rank']) if m['first_citation_rank'] else '-'
        pawc = f"{m['pawc']:.1f}" if m['pawc'] > 0 else '-'
        cr = f"{m['citation_rate']:.0f}%" if m['citation_rate'] is not None else '-'
        print(f"{r['query'][:58]:<60} {vis:>4} {m['som']:>5.1f}% {rank:>5} {pawc:>7} {cr:>6}")
    else:
        print(f"{r['query'][:58]:<60} {'ERR':>4}")